In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

# Loading the pre-computed planforms and the fuselage

In [19]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [5]:
# --- Import Onshape pull utilities ---
sys.path.append(os.path.abspath(os.getcwd()))
from onshape_pull import (
    fetch_variable_studio, fetch_measurement_features,
    evaluate_measurements, load_cached_masses, fetch_mass_properties,
    compute_cg_scenarios, lookup_var, lookup_meas,
    UPDATE_MASSES,
)

# --- Pull data from Onshape ---
variables = fetch_variable_studio()
meas_names = fetch_measurement_features()
measurements = evaluate_measurements(meas_names)
components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
cg_data = compute_cg_scenarios(components)

# --- Z offset (axle datum) ---
Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
            + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# --- Build drag components ---
engine_bay = Bay(
    surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
    length=0.172,                       # 172 mm (hardcoded, not in Onshape)
    diameter=lookup_var(variables, "engine_diameter")[0],
)

Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
nose_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
)

Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
main_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
)

fuselage = Fuselage(
    surface_wetted=lookup_meas(measurements, "Wetted_Area"),
    length_total=lookup_var(variables, "FuselageLength")[0],
    diameter_max=lookup_var(variables, "FuselageHeight")[0],
    upsweep=0.0,
    base_area=lookup_meas(measurements, "Base_Area"),
)

# --- X-position helpers ---
WingPortDistance, _, _ = lookup_var(variables, "WingPortDistance")
WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# --- Fixed parameters ---
fixed = Fixed(
    mass=cg_data["mass"],
    fuel_mass=cg_data["fuel_mass"],
    x_cg_min=cg_data["x_cg_min"],
    x_cg_max=cg_data["x_cg_max"],
    x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
    z_cg=cg_data["z_cg_full"] + z_offset,
    z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
    z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
    x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
    x_LE_wing=WingPortDistance + WingPortWidth / 2,
    x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
    x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
    x_main_gear=lookup_meas(measurements, "x_main_gear"),
    y_main_gear=0.419,
    fuselage=fuselage,
    nose_gear=nose_gear,
    main_gear=main_gear,
    engine_bay=engine_bay,
)

  → Found 78 occurrences, fetching mass properties...
    ... processed 10/78 occurrences
    ... processed 20/78 occurrences
    ... processed 30/78 occurrences
    ... processed 40/78 occurrences
    ... processed 50/78 occurrences
    ... processed 60/78 occurrences
    ... processed 70/78 occurrences
  → Mass data cached to mass_cache.json


In [6]:
with open("pickles/fixed_pickle.pcl", "wb") as f:
    pickle.dump(fixed, f)

In [7]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [8]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [20]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

Iteration 0: Sh_S=0.527177314066423, lh=2.3952060372181494, x_ac_h=3.315079018295956, x_cg=(0.223115669701992, 0.4297945838394097), Local_ac: 0.06367981927798369
Iteration 1: Sh_S=0.4577993457355067, lh=2.734237639143065, x_ac_h=3.6541106202208717, x_cg=(0.23066525834855411, 0.4373441724859718), Local_ac: 0.14621104420304512
Iteration 2: Sh_S=0.4654407489863632, lh=2.6933222057110573, x_ac_h=3.613195186788864, x_cg=(0.2293918326236691, 0.43607074676108676), Local_ac: 0.13625090827379568
Iteration 3: Sh_S=0.4645635550247432, lh=2.6979740827261574, x_ac_h=3.617847063803964, x_cg=(0.22953053681528815, 0.43620945095270586), Local_ac: 0.137383325151978
Iteration 4: Sh_S=0.46466377306751155, lh=2.69744202112428, x_ac_h=3.617315002202087, x_cg=(0.229514595138595, 0.4361935092760127), Local_ac: 0.13725380420027536


In [21]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.1506793601301967, 0.02560347511843066, 0.16601536408037712, 0.15290186804376898, 0.4717757799637618, 0.3373609380643771, 0.25939799776077316, 0.16903353183800476, 0.1506793601301967, 0.02560347511843066, 0.16601536408037712, 0.15290186804376898, 0.4717757799637618, 0.3373609380643771, 0.25939799776077316, 0.16903353183800476, 0.1506793601301967, 0.02560347511843066, 0.16601536408037712, 0.15290186804376898, 0.4717757799637618, 0.3373609380643771, 0.25939799776077316, 0.16903353183800476, 0.24035998946578843, 0.1180993774824261, 0.24035998946578843, 0.2543799679956308, 0.46465804506981045, 0.33721958001818664, 0.3224161500313257, 0.22016637521042484, 0.24035998946578843, 0.1180993774824261, 0.24035998946578843, 0.2543799679956308, 0.46465804506981045, 0.33721958001818664, 0.3224161500313257, 0.22016637521042484, 0.24035998946578843, 0.1180993774824261, 0.24035998946578843, 0.2543799679956308, 0.46465804506981045, 0.33721958001818664, 0.3224161500313257, 0.22016637521042484, 0.2637405

# Checking if reuirements are met

In [17]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [22]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 11.000000001437998 kg
Fuel required: 7.70886148548979 kg
Difference: 3.291138515948208 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.709648070115518 kg
Difference: 3.29035193132248 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.695942964832593 kg
Difference: 3.304057036605405 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.646281300958993 kg
Difference: 3.353718700479005 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.403330737373577 kg
Difference: 3.5966692640644213 kg
all constraints satisfied
ac mass: 41.06416100580495, 0.7845238526961408
MainWing: AR=5.0, tc=0.06, sweep=14.999999999999998 deg, cmac=-0.15
Failed: ['Empennage Requirement']

Fuel available: 11.000000001437998 kg
Fuel required: 7.573005477491933 kg
Difference: 3.426994523946065 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 